In [ ]:
import torch
import json

In [ ]:
import json
from sklearn.metrics import fbeta_score, f1_score, precision_recall_fscore_support, accuracy_score, matthews_corrcoef, classification_report

def accuracy(preds, labels):
    correct = sum(int(pred == label) for pred, label in zip(preds, labels))
    total = len(labels)
    accuracy = correct / total
    return accuracy


def precision(preds, labels):
    true_positives = sum(int(pred == 1 and label == 1) for pred, label in zip(preds, labels))
    false_positives = sum(int(pred == 1 and label == 0) for pred, label in zip(preds, labels))
    precision = true_positives / (true_positives + false_positives)
    return precision


def recall(preds, labels):
    true_positives = sum(int(pred == 1 and label == 1) for pred, label in zip(preds, labels))
    false_negatives = sum(int(pred == 0 and label == 1) for pred, label in zip(preds, labels))
    recall = true_positives / (true_positives + false_negatives)
    return recall

def evaluator(preds, labels, raw_pred_scores=None):
    pos_correct_count, neg_correct_count = 0,0
    for i in range(len(preds)):
        if preds[i] == labels[i]:
            if labels[i] == 1:
                pos_correct_count +=1
            elif labels[i] == 0:
                neg_correct_count +=1

    print('accuracy: ', round(accuracy(preds, labels), 4))
    print('precision: ', round(precision(preds, labels), 4))
    print('recall: ', round(recall(preds, labels), 4))
    print('F0.5: ', fbeta_score(labels, preds, average=None, beta=0.5)[1])
    print('F1: ', f1_score(labels, preds))

In [ ]:

reference_path = r'/root/autodl-tmp/Blip2CC/.cache/coco/annotations/coco_karpathy_test.json'
with open(reference_path, 'r') as j:
    gt_raw = json.load(j)
test_path = r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20250320205/test_epochbest.json'
with open(test_path, 'r') as j:
    test_data = json.load(j)

# test_path = r'/root/autodl-tmp/Blip2CC/promptcc-A2B.txt'
# with open(test_path, 'r') as f:
#     test_data = f.readlines()
#     for i in range(len(test_data)):
#         test_data[i] = test_data[i].rstrip('\n')

In [ ]:
# 计算分类指标
nochange_caps = ["there is no difference", "the two scenes seem identical", "the scene is the same as before", "no change has occurred", "almost nothing has changed"]
pre = []
truth = []
for gt, test in list(zip(gt_raw, test_data)):
    label = gt['changeflag']
    test_cap = test['caption']
    # test_cap = test
    if test_cap in nochange_caps:
        pre.append(0)
        truth.append(label)
    else:
        pre.append(1)
        truth.append(label)

evaluator(pre, truth)

In [ ]:
# 准确率、精确率
nochange_caps = ["there is no difference", "the two scenes seem identical", "the scene is the same as before", "no change has occurred", "almost nothing has changed"]
test_nochange_cap_num = 0
t_nochange_num = 0
test_change_cap_num = 0
t_change_num = 0
for gt, test in list(zip(gt_raw, test_data)):
    label = gt['changeflag']
    test_cap = test['caption']
    # test_cap = test
    if test_cap in nochange_caps:
        test_nochange_cap_num += 1
        if label==0:
            t_nochange_num += 1
    else:
        test_change_cap_num += 1
        if label==1:
            t_change_num += 1

nochange_precision = t_nochange_num/test_nochange_cap_num
change_precision = t_change_num/test_change_cap_num
print("Nochange Precision:", nochange_precision) 
print("Change Precision:", change_precision)
print("Precision:", (t_nochange_num+t_change_num)/len(gt_raw))

In [ ]:
# 召回率
nochange_caps = ["there is no difference", "the two scenes seem identical", "the scene is the same as before", "no change has occurred", "almost nothing has changed"]
nochange_cap_num = 0
t_nochange_num = 0
change_cap_num = 0
t_change_num = 0
for gt, test in list(zip(gt_raw, test_data)):
    label = gt['changeflag']
    # test_cap = test
    test_cap = test['caption']
    if label==0:
        nochange_cap_num += 1
        if test_cap in nochange_caps:
            t_nochange_num += 1
    else:
        change_cap_num += 1
        if test_cap not in nochange_caps:
            t_change_num += 1

nochange_recall = t_nochange_num/nochange_cap_num
change_recal = t_change_num/change_cap_num
print("Nochange Recall:", nochange_recall) 
print("Change Recall:", change_recal)
# print("Precision:", (t_nochange_num+t_change_num)/len(gt_raw))

In [ ]:
import json
from transformers import GPT2Tokenizer
import torch
gpt2_type = '/root/autodl-tmp/gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
reference_path = r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20240813221/test_epochbest.json'
with open(reference_path, 'r') as j:
    gt_raw = json.load(j)
num = 0
for item in gt_raw:
    sentence = item['caption']
    answer = torch.tensor(tokenizer.encode(sentence)).tolist()
    # print(answer)
    if len(answer)>num:
        num = len(answer)
        if num==29:
            print(sentence)
print(num)

In [ ]:
import torch
from model.modeling_llama import LlamaForCausalLM
llm_model = LlamaForCausalLM.from_pretrained(
            r'/root/autodl-tmp/bert-base-uncased', torch_dtype=torch.float16
        )

print(llm_model)

In [ ]:
from model.Qformer import BertConfig, BertLMHeadModel
Qformer = BertLMHeadModel.from_pretrained(
            "/root/autodl-tmp/bert-base-uncased"
        )
print(Qformer)

In [ ]:
import json
with open(r'/root/autodl-tmp/Blip2CC/.cache/coco/annotations/coco_karpathy_train.json', 'r') as f:
    data = json.load(f)
di = data[0:10]
for d in di:
    print(d)